# Graphicalizing Aesop's Fables

This notebook downloads the public-domain Project Gutenberg text for [Æsop's Fables #53103](https://www.gutenberg.org/ebooks/53103), splits it into tale-sized documents, and sends those documents through the ontology-guided semantic pipeline.

The notebook uses the package's default OpenAI `gpt-4.1-mini` client. Set `OPENAI_API_KEY` in the environment before running the model cell. The key is read by the OpenAI SDK and is never stored in this notebook.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from textwrap import wrap

from IPython.display import display

from semantic_graphicalizer import SemanticGraphicalizer, load_aesop_fables

ROOT = Path.cwd()
if not (ROOT / 'configs').exists():
    ROOT = ROOT.parent

stories = load_aesop_fables(limit=2, cache_dir=ROOT / 'data' / 'raw')
[(story.splitlines()[0], len(story)) for story in stories]

[('THE DAW IN BORROWED FEATHERS', 859), ('THE SUN AND THE WIND', 972)]

In [2]:
len(stories), [story.splitlines()[0] for story in stories]

(2, ['THE DAW IN BORROWED FEATHERS', 'THE SUN AND THE WIND'])

## OpenAI model

The transformer creates the default `OpenAIModelClient` when `model` is omitted. It uses `gpt-4.1-mini` through the Responses API and reads `OPENAI_API_KEY` from the environment.

In [8]:
from textwrap import wrap

graphicalizer = SemanticGraphicalizer(
    ontology=ROOT / 'configs' / 'ontologies' / 'aesop.yaml',
    prompts=ROOT / 'configs' / 'prompts' / 'aesop.yaml',
)


graphicalizer.fit(stories)
traces = []

for index, story in enumerate(stories, start=1):
    trace = graphicalizer.transform_with_trace([story])[0]
    traces.append(trace)
    title = story.splitlines()[0]
    print(f"\nDocument {index}/{len(stories)}: {title}")
    print("\n".join(wrap(story, width=80)))
    print(f"ID: {trace.document_id}")
    print(f"Propositions: {len(trace.propositions)} | Triples: {len(trace.triples)} | "
          f"Nodes: {trace.graph.number_of_nodes()} | Edges: {trace.graph.number_of_edges()}")
    print("-" * 80)
    for proposition in trace.propositions:
        print(f"- {proposition.text}")
    print("\nGraph:")
    display(graphicalizer.display(trace.graph, mode="text"))
    print("=" * 80)


[SemanticGraphicalizer] ready: model=OpenAIModelClient, ontology=aesop-narrative, domain=aesop-narrative
[document-71c1b92da992] processing document: chars=859
[document-71c1b92da992] segment: 1 -> 1 | 0.0 ms | input_chars=859, chunk_chars=859
[document-71c1b92da992 chunk-0] compiling semantic stages
[document-71c1b92da992 chunk-0] summarize: 1 -> 1 | 3.0 s | input_chars=859, output_chars=498
[document-71c1b92da992 chunk-0] normalize: 1 -> 1 | 2.0 s | input_chars=498, output_chars=582
[document-71c1b92da992 chunk-0] decompose: 1 -> 11 | 5.7 s
[document-71c1b92da992 chunk-0] triple: 11 -> 11 | 10.1 s
[document-71c1b92da992] integrate: 11 -> 11 | 0.8 ms | nodes=14, edges=11
[document-71c1b92da992] total: 1 -> 1 | 20.9 s | chunks=1, propositions=11, triples=11, nodes=14, edges=11

Document 1/2: THE DAW IN BORROWED FEATHERS
['THE DAW IN BORROWED FEATHERS   [Illustration: The Daw in Borrowed Feathers]  A', 'conceited jackdaw was vain enough to imagine that he wanted nothing but the', 'colou

Animal: jackdaw
    has_trait: Trait: conceit
    wants: Goal: to appear as beautiful as peacocks
    performs: Action: dressing himself in the peacocks' finest feathers
    performs: Action: pretending to be a peacock
    performs: Action: performing an unnatural strut when trying to join the peacocks
    causes: Action: the peacocks recognizing his disguise
    performs: Action: returning to his own kind
    interacts_with: Animal: his companions who rejected him
Animal: peacocks
    performs: Action: attacking the jackdaw
    performs: Action: removing the jackdaw's borrowed feathers
Character: narrative
    teaches: Moral: vanity and pretending to be something one is not results in rejection and trouble

[document-127171afb681] processing document: chars=972
[document-127171afb681] segment: 1 -> 1 | 0.1 ms | input_chars=972, chunk_chars=972
[document-127171afb681 chunk-0] compiling semantic stages
[document-127171afb681 chunk-0] summarize: 1 -> 1 | 1.2 s | input_chars=972, output_chars=406
[document-127171afb681 chunk-0] normalize: 1 -> 1 | 1.3 s | input_chars=406, output_chars=600
[document-127171afb681 chunk-0] decompose: 1 -> 7 | 12.6 s
[document-127171afb681 chunk-0] triple: 7 -> 8 | 10.6 s
[document-127171afb681] integrate: 8 -> 8 | 0.5 ms | nodes=10, edges=8
[document-127171afb681] total: 1 -> 1 | 25.7 s | chunks=1, propositions=7, triples=8, nodes=10, edges=8

Document 2/2: THE SUN AND THE WIND
['THE SUN AND THE WIND   [Illustration: The Sun and The Wind]  [Illustrations]  A', 'dispute once arose between the Sun and the Wind, which was the stronger of the', 'two, and they agreed to count this as proof, that whichever soonest made a', 'traveller take off his cloak, should be held

Character: The Sun
    interacts_with: Character: The Wind
    wants: Goal: demonstrate their strength by causing a traveller to remove his cloak
    performs: Action: shining warmly
Character: The Wind
    wants: Goal: demonstrate their strength by causing a traveller to remove his cloak
    performs: Action: blowing fiercely
Human: The traveller
    performs: Action: wrapping the cloak tighter
Action: shining warmly
    causes: Goal: traveller removing his cloak
Character: The tale
    teaches: Moral: gentle persuasion is more effective than force and fury

In [17]:
graphicalizer.display(
    traces[0].graph,
    mode="dynamic",
    charge_strength=0,
    link_distance=80,
    component_spacing=30,
    component_strength=0.15,
)